## Cross-validation
* Cross-validation is a data-splitting technique used to get an honest, reliable estimate of a model's performance without relying on a single, lucky or unlucky train/validation split.

## Stratified K-fold
* a variation of K-fold cross-validation that ensures each fold contains the exact same percentage of target categories as the complete dataset.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

In [2]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

In [3]:
print(df.shape)

(7043, 21)


In [4]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["churn"] = (df["Churn"] == "Yes").astype(int)

num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = ["Contract", "InternetService", "PaymentMethod",
           "PaperlessBilling", "Partner", "Dependents", "TechSupport"]

X = pd.get_dummies(df[num + cat], columns=cat, dtype=int)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train
)

print(X.shape, X.isna().sum().sum(), round(y.mean(), 4))
print(X_train.shape, X_test.shape, X_tr.shape, X_val.shape)
print(y_val.sum(), y_test.sum())

(7043, 23) 11 0.2654
(5634, 23) (1409, 23) (4225, 23) (1409, 23)
374 374


In [5]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

In [6]:
def make_lr():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

aucs = []
for seed in range(10):
    Xa, Xb, ya, yb = train_test_split(
        X_train, y_train, test_size=0.25, random_state=seed, stratify=y_train
    )
    m = make_lr().fit(Xa, ya)
    aucs.append(roc_auc_score(yb, m.predict_proba(Xb)[:, 1]))
aucs = np.array(aucs)
print(aucs.round(4))
print("min", aucs.min().round(4), "max", aucs.max().round(4), "range", (aucs.max() - aucs.min()).round(4))

[0.8403 0.8208 0.8416 0.8387 0.8441 0.8593 0.8523 0.8275 0.8436 0.8266]
min 0.8208 max 0.8593 range 0.0385


In [7]:
kf = KFold(n_splits=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Plain KFold:", [round(float(y_train.iloc[v].mean()), 3) for _, v in kf.split(X_train, y_train)])
print("Stratified:", [round(float(y_train.iloc[v].mean()), 3) for _, v in skf.split(X_train, y_train)])
print("train/val rows in the first 2 folds:", [(len(t), len(v)) for t, v in skf.split(X_train, y_train)][:2])

Plain KFold: [0.258, 0.252, 0.284, 0.271, 0.262]
Stratified: [0.265, 0.265, 0.265, 0.265, 0.266]
train/val rows in the first 2 folds: [(4507, 1127), (4507, 1127)]


In [8]:
scores = cross_val_score(make_lr(), X_train, y_train, cv=skf, scoring="roc_auc")
print(scores.round(4), "mean", scores.mean().round(4), "std", scores.std().round(4))

[0.8434 0.8249 0.8327 0.8579 0.8523] mean 0.8422 std 0.0122


In [9]:
def make_rf(**kw):
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("model", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, **kw))])

models = {"logistic regression": make_lr(),
          "forest default": make_rf(),
          "forest tuned (depth 5, leaf 5)": make_rf(max_depth=5, min_samples_leaf=5)}
for name, m in models.items():
    s = cross_val_score(m, X_train, y_train, cv=skf, scoring="roc_auc")
    print(f"{name:32s} mean {s.mean():.4f}  std {s.std():.4f}  folds {s.round(4)}")

logistic regression              mean 0.8422  std 0.0122  folds [0.8434 0.8249 0.8327 0.8579 0.8523]
forest default                   mean 0.8102  std 0.0122  folds [0.8097 0.7992 0.7963 0.8158 0.8301]
forest tuned (depth 5, leaf 5)   mean 0.8443  std 0.0106  folds [0.8381 0.8322 0.8373 0.8585 0.8555]


In [10]:
import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint

In [18]:
def cv(model):
    s = cross_val_score(model, X_train, y_train, cv=skf, scoring="roc_auc")
    return s.mean(), s.std()

def pipe(scaler, model):
    steps = [("imp", SimpleImputer(strategy="median"))]
    if scaler is not None:
        steps.append(("sc", scaler))
    steps.append(("model", model))
    return Pipeline(steps)

In [19]:
print(X_train[num].describe().loc[["min", "max", "mean", "std"]].round(1))

      tenure  MonthlyCharges  TotalCharges  SeniorCitizen
min      0.0            18.4          18.8            0.0
max     72.0           118.8        8684.8            1.0
mean    32.5            64.9        2302.6            0.2
std     24.6            30.1        2279.2            0.4


In [26]:
models = {
    "logistic regression": lambda: LogisticRegression(max_iter=1000),
    "KNN (k=15)": lambda: KNeighborsClassifier(n_neighbors=15),
    "forest (depth 5)": lambda: RandomForestClassifier(n_estimators=300, max_depth=5,
                                     min_samples_leaf=5, random_state=42, n_jobs=-1),
}
for mname, mk in models.items():
    for sname, sc in [("none", None), ("Standard", StandardScaler()), ("MinMax", MinMaxScaler())]:
        m, s = cv(pipe(sc, mk()))
        print(f"{mname:20s} {sname:9s} {m:.4f} ± {s:.4f}")

c:\Users\hknia\ai-ml-30days\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logistic regression  none      0.8423 ± 0.0123
logistic regression  Standard  0.8422 ± 0.0122
logistic regression  MinMax    0.8417 ± 0.0119
KNN (k=15)           none      0.7822 ± 0.0088
KNN (k=15)           Standard  0.8128 ± 0.0115
KNN (k=15)           MinMax    0.8109 ± 0.0119
forest (depth 5)     none      0.8443 ± 0.0106
forest (depth 5)     Standard  0.8443 ± 0.0106
forest (depth 5)     MinMax    0.8443 ± 0.0106


In [27]:
for k in [3, 5, 8, 12, 16, 23]:
    p = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("sel", SelectKBest(f_classif, k=k)),
        ("model", LogisticRegression(max_iter=1000))
    ])
    m , s = cv(p)
    print(f"k={k:2d}  {m:.4f} ± {s:.4f}")

k= 3  0.8074 ± 0.0091
k= 5  0.8352 ± 0.0128
k= 8  0.8384 ± 0.0123
k=12  0.8417 ± 0.0127
k=16  0.8422 ± 0.0125
k=23  0.8422 ± 0.0122


In [25]:
grid = {"model__max_depth": [3, 5, 8, None], "model__min_samples_leaf": [1, 5, 20]}
gs = GridSearchCV(make_rf(), grid, cv=skf, scoring="roc_auc")
t0 = time.time(); gs.fit(X_train, y_train)
print(f"{time.time()-t0:.0f}s", gs.best_params_, round(gs.best_score_, 4))
res = pd.DataFrame(gs.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf",
                                    "mean_test_score", "std_test_score", "rank_test_score"]]
print(res.sort_values("rank_test_score").round(4).to_string(index=False))

c:\Users\hknia\ai-ml-30days\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\hknia\ai-ml-30days\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\hknia\ai-ml-30days\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\hknia\ai-ml-30days\.venv\Lib\site-packages\sklearn\utils\paralle

30s {'model__max_depth': 8, 'model__min_samples_leaf': 20} 0.8448
param_model__max_depth  param_model__min_samples_leaf  mean_test_score  std_test_score  rank_test_score
                     8                             20           0.8448          0.0106                1
                  None                             20           0.8448          0.0108                2
                     5                              5           0.8443          0.0106                3
                     5                             20           0.8442          0.0105                4
                     5                              1           0.8442          0.0106                5
                     8                              5           0.8436          0.0104                6
                     8                              1           0.8421          0.0107                7
                     3                              1           0.8404          0.0099                

In [23]:
dist = {"model__max_depth": randint(2, 12), "model__min_samples_leaf": randint(1, 40),
        "model__max_features": [0.3, 0.5, "sqrt"]}
rs = RandomizedSearchCV(make_rf(), dist, n_iter=12, cv=skf, scoring="roc_auc", random_state=42)
t0 = time.time(); rs.fit(X_train, y_train)
print(f"{time.time()-t0:.0f}s", rs.best_params_, round(rs.best_score_, 4))

30s {'model__max_depth': 10, 'model__max_features': 0.3, 'model__min_samples_leaf': 27} 0.8457
